In [ ]:
import pandas as pd
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


pd.set_option("display.max_colwidth", None)

In [23]:
data = pd.read_csv("Finetuning/data/processed/data.csv")
data.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0001ea8717f6de06,Thank you for understanding. I think very highly of you and would not revert without discussion.,0,0,0,0,0,0
1,000247e83dcc1211,:Dear god this site is horrible.,0,0,0,0,0,0
2,0002f87b16116a7f,"""::: Somebody will invariably try to add Religion? Really?? You mean, the way people have invariably kept adding """"Religion"""" to the Samuel Beckett infobox? And why do you bother bringing up the long-dead completely non-existent """"Influences"""" issue? You're just flailing, making up crap on the fly. \n ::: For comparison, the only explicit acknowledgement in the entire Amos Oz article that he is personally Jewish is in the categories! \n\n """,0,0,0,0,0,0
3,0003e1cccfd5a40a,""" \n\n It says it right there that it IS a type. The """"Type"""" of institution is needed in this case because there are three levels of SUNY schools: \n -University Centers and Doctoral Granting Institutions \n -State Colleges \n -Community Colleges. \n\n It is needed in this case to clarify that UB is a SUNY Center. It says it even in Binghamton University, University at Albany, State University of New York, and Stony Brook University. Stop trying to say it's not because I am totally right in this case.""",0,0,0,0,0,0
4,00059ace3e3e9a53,""" \n\n == Before adding a new product to the list, make sure it's relevant == \n\n Before adding a new product to the list, make sure it has a wikipedia entry already, """"proving"""" it's relevance and giving the reader the possibility to read more about it. \n Otherwise it could be subject to deletion. See this article's revision history.""",0,0,0,0,0,0


In [ ]:
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
y = data[label_cols].values

msss_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, test_dev_idx = next(msss_1.split(np.zeros(len(data)), y))

train_df = data.iloc[train_idx].reset_index(drop=True)
test_dev_df = data.iloc[test_dev_idx].reset_index(drop=True)

y_test_dev = test_dev_df[label_cols].values
msss_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
test_idx, dev_idx = next(msss_2.split(np.zeros(len(y_test_dev)), y_test_dev))

test_df = data.iloc[test_idx].reset_index(drop=True)
dev_df = data.iloc[dev_idx].reset_index(drop=True)

print(len(train_df), len(test_df), len(dev_df))

def pos_rate(df):
    return df[label_cols].mean()

rates = pd.DataFrame({
    "full": pos_rate(data),
    "train": pos_rate(train_df),
    "dev": pos_rate(dev_df),
    "test": pos_rate(test_df),
})
print((rates * 100).round(2))


201194 11177 11178
               full  train   dev  test
toxic          9.57   9.57  9.22  9.50
severe_toxic   0.88   0.88  0.55  0.54
obscene        5.43   5.43  5.53  5.76
threat         0.31   0.31  0.41  0.29
insult         5.06   5.06  5.05  5.31
identity_hate  0.95   0.95  0.98  1.20


In [27]:
for label in label_cols:
    print(f"======={label}======")
    print(data[[label]].value_counts())

=======toxic======
toxic
0        202165
1         21384
Name: count, dtype: int64
=======severe_toxic======
severe_toxic
0               221587
1                 1962
Name: count, dtype: int64
=======obscene======
obscene
0          211409
1           12140
Name: count, dtype: int64
=======threat======
threat
0         222860
1            689
Name: count, dtype: int64
=======insult======
insult
0         212245
1          11304
Name: count, dtype: int64
=======identity_hate======
identity_hate
0                221432
1                  2117
Name: count, dtype: int64


In [ ]:
train_df.to_csv("Finetuning/data/processed/train.csv", index=False)
dev_df.to_csv("Finetuning/data/processed/dev.csv", index=False)
test_df.to_csv("Finetuning/data/processed/test.csv", index=False)

We can see a clear imbalance especially in the severe_toxic, threat, and indentity_hate labels.